# Notebook 05: Evaluation, False Positives & Confusion Matrix
## Smart CCTV AI Safety & Identification Platform

This notebook covers:
1. Comprehensive performance metrics: Accuracy, Precision, Recall, F1-Score
2. Multi-class Confusion Matrix Visualization
3. Analysis of Fall False Positives and False Negatives
4. Impact of Temporal State Machine Verification on reducing false alarms

In [ ]:
import sys
sys.path.append('..')

import pickle
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from src.features.feature_extractor import FEATURE_COLUMNS

# Load test data and trained model
df = pd.read_csv('../data/raw/fall_activity_dataset.csv')
with open('../models/fall_detection_model.pkl', 'rb') as f:
    model = pickle.load(f)

X = np.array(df[FEATURE_COLUMNS], dtype=np.float32)
y = np.array(df['label'])
y_pred = model.predict(X)

In [ ]:
# 1. Classification Metrics
print("=== Overall Classification Report ===")
print(classification_report(y, y_pred))

# 2. Confusion Matrix Heatmap
labels = sorted(list(np.unique(y)))
cm = confusion_matrix(y, y_pred, labels=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Multi-Class Confusion Matrix (Activity & Fall Detection)')
plt.xlabel('Predicted Label')
plt.ylabel('Ground Truth Label')
plt.show()

In [ ]:
# 3. Discussion on Fall Sensitivity vs Specificity
fall_idx = labels.index('falling')
tp = cm[fall_idx, fall_idx]
fn = np.sum(cm[fall_idx, :]) - tp
fp = np.sum(cm[:, fall_idx]) - tp
tn = np.sum(cm) - (tp + fn + fp)

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0

print(f"Fall Sensitivity (Recall): {sensitivity*100:.2f}%")
print(f"Fall Specificity:        {specificity*100:.2f}%")
print(f"Fall Precision:          {precision*100:.2f}%")
print(f"False Positives (FP):    {fp}")
print(f"False Negatives (FN):    {fn}")